![img](https://github.com/JuliaLang/julia/raw/master/doc/src/assets/logo.svg)![img](https://avatars.githubusercontent.com/u/7346142?s=200&v=4)

# 1. Installation and Setup

JuliaGPU packages are easy to install: Just do `Pkg.add("Package Name")` to install the package.

In [ ]:
using Pkg
Pkg.add(["BenchmarkTools", "KernelAbstractions", "CUDA", "AMDGPU", "Metal", "CairoMakie"])

In [ ]:
using BenchmarkTools, LinearAlgebra, KernelAbstractions

In [ ]:
Pkg.status()

## 1.1. Selecting a backend

GPU backend packages provide bindings to NVIDIA's CUDA or respective other vendor. CUDA.jl provides all of the compiler and runtime logic needed to program NVIDIA GPUs; the only thing you need to provide is a functional NVIDIA driver (which most HPC systems already have installed and configured), but you don't need to install the CUDA toolkit! CUDA.jl downloads one if it's not already available on your system (again, HPC systems usually have this provided for you):

In [ ]:
# add all options for different backends and array types
DEV = :AMDGPU

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal 
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
else DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
end
#allocate(backend, Float32, res, res)

# 2. Example: Matrix-Matrix Multiplication



## 2.1. CPU matrix multiplication

Matrix multiplication is a mainstay of all kinds of applications, so we should be able to implement this in Julia with ease. Let's first take a look at doing this on the CPU:



In [ ]:
# Allocate our random matrix inputs and zero'd output
matrix_size = 1024
A = rand(matrix_size, matrix_size)
B = rand(matrix_size, matrix_size)
C = zeros(matrix_size, matrix_size)

A



The three nested loops implementation of matrix multiplication is easy to express on the CPU:



In [ ]:
function MatrixMultiplication!(C, A, B)
    for i in 1:size(C, 1)
        for j in 1:size(C, 2)
            C[i, j] = 0
            for k in 1:size(A, 2)
                C[i, j] += A[i, k] * B[k, j]
            end
        end
    end
end



In [ ]:
MatrixMultiplication!(C, A, B)
C



Of course, Julia has this one built-in already (it calls BLAS):



In [ ]:
C = A * B



Our implementation performs quite a bit worse than Julia's (OpenBLAS), but that's OK - we haven't really optimized it at all, since this is just a tutorial:



In [ ]:
@benchmark MatrixMultiplication!(C, A, B)



In [ ]:
@benchmark A * B



## 2.2. Benchmarking across matrix sizes

The timings above are one matrix, 1024×1024. The same multiply gets much more expensive as the side length grows, so we time it at `N = 16, 64, 256, 1024, 2048`.

`benchmark_matmul!` does not know whether the multiply runs on the CPU or the GPU. `setup(n)` builds the `n×n` inputs however it needs to, then returns a function with no arguments. `@belapsed` times only that function, so allocation and the copy onto the device stay outside the measurement. Call the function once before timing so compilation is not part of the result. A GPU setup puts `KernelAbstractions.synchronize` inside the function it returns. Otherwise the timer would stop at the launch, before the kernel finishes.

Each call appends one series to `matmul_results`. `plot_matmul` draws every series stored there. Later kernels push another series and call `plot_matmul` again, which adds a line to the same axes.



In [ ]:
using CairoMakie

matmul_sizes = [16, 64, 256, 1024, 2048]
matmul_results = Pair{String, Vector{Float64}}[]

function benchmark_matmul!(results, label, sizes, setup)
    # Replacing a label lets the cell be run again without stacking duplicate lines.
    filter!(pair -> pair.first != label, results)
    times = Float64[]
    for n in sizes
        bench = setup(n)
        bench()
        push!(times, @belapsed $bench())
    end
    push!(results, label => times)
    return times
end

function plot_matmul(sizes, results)
    fig = Figure()
    ax = Axis(fig[1, 1],
        xlabel = "matrix size N",
        ylabel = "time (s)",
        xscale = log2,
        yscale = log10,
        xticks = (sizes, string.(sizes)),
    )
    for (label, times) in results
        lines!(ax, sizes, times, label = label)
        scatter!(ax, sizes, times)
    end
    axislegend(ax, position = :lt)
    fig
end



In [ ]:
benchmark_matmul!(matmul_results, "CPU MatrixMultiplication!", matmul_sizes) do n
    A = rand(n, n)
    B = rand(n, n)
    C = zeros(n, n)
    () -> MatrixMultiplication!(C, A, B)
end

benchmark_matmul!(matmul_results, "CPU * (OpenBLAS)", matmul_sizes) do n
    A = rand(n, n)
    B = rand(n, n)
    () -> A * B
end

plot_matmul(matmul_sizes, matmul_results)



## 2.3. Naive GPU matrix multiplication

Ok, let's now implement matrix multiplication on the GPU. We move `A` and `B` to the device and write a kernel in the same style as `vadd_kernel!`: one thread per output element, grouped into blocks that each run on one streaming multiprocessor.



In [ ]:
# Allocate device memory for A and B, then copy the CPU matrices over,
# just as we did for the vectors in vadd_kernel!.
# KernelAbstractions.zeros allocates DC and fills it with zeros.
DA = allocate(Backend, eltype(A), size(A)...)
copyto!(DA, A)
DB = allocate(Backend, eltype(B), size(B)...)
copyto!(DB, B)
DC = KernelAbstractions.zeros(Backend, eltype(A), size(A)...)

DA



As with the vectors, we can multiply `DA` by `DB` with `*`. Multiple dispatch forwards that call to the backend's matrix library (cuBLAS when `Backend` is CUDA).



In [ ]:
DC = DA * DB



In [ ]:
@kernel function MatrixMultiplication_kernel!(C, A, B)
    # Global row and column across every block of the launch.
    # In 2D this is the same idea as @index(Global) in vadd_kernel!:
    # the thread's place in its block, plus that block's offset in the grid.
    row, col = @index(Global, NTuple)

    # A zero of the same type as C's elements (keeps the sum type-stable)
    sum = zero(eltype(C))

    # The last blocks can extend past the matrix, the same bounds check as in vadd_kernel!.
    if row <= size(A, 1) && col <= size(B, 2)
        for i in 1:size(A, 2)
            # @inbounds skips Julia's array bounds checks. Wrong indices fault or return bad values.
            @inbounds sum += A[row, i] * B[i, col]
        end
        @inbounds C[row, col] = sum
    end

    return
end



In [ ]:
# Each block is a 32×32 tile: 1024 threads, the most one SM accepts in a block.
# That is the same cap that made the 10240-element vadd_kernel! use more than one block.
# ndrange is the whole matrix, so the number of blocks on each axis is matrix_size ÷ 32.
workgroupsize = (32, 32)
kernel! = MatrixMultiplication_kernel!(Backend, workgroupsize)
kernel!(DC, DA, DB, ndrange=size(DC))

KernelAbstractions.synchronize(Backend)
DC



The library `DA * DB` is far faster than `MatrixMultiplication_kernel!`:



In [ ]:
@benchmark DA * DB



In [ ]:
@benchmark begin
    kernel!(DC, DA, DB, ndrange=size(DC))
    KernelAbstractions.synchronize(Backend)
end



Our `MatrixMultiplication_kernel!` is slower than `DA * DB` because the kernel is the naive algorithm. Every thread walks the full inner dimension and reads `DA` and `DB` from global memory on every product.

The library `*` tiles the matrices. Each block loads a piece of `DA` and `DB` into the shared memory from the thread-block picture, and the threads on that SM reuse those values. Optimized kernels also use vendor matrix instructions (WMMA on NVIDIA).

Each 32×32 block of `MatrixMultiplication_kernel!` owns one tile of `DC`, and it never copies the matching tiles into that shared memory, so every product goes back to global memory. The next section loads those tiles into shared memory.



Same sizes, now with `MatrixMultiplication_kernel!`. The workgroup is 32×32, or the side of the matrix when `N` is smaller than 32, so a 16×16 problem does not ask for more threads than it has elements. `plot_matmul` draws this series on the axes above.



In [ ]:
benchmark_matmul!(matmul_results, "GPU MatrixMultiplication_kernel!", matmul_sizes) do n
    A = rand(n, n)
    B = rand(n, n)
    DA = copyto!(allocate(Backend, Float64, n, n), A)
    DB = copyto!(allocate(Backend, Float64, n, n), B)
    DC = KernelAbstractions.zeros(Backend, Float64, n, n)
    w = min(32, n)
    k! = MatrixMultiplication_kernel!(Backend, (w, w))
    function bench()
        k!(DC, DA, DB, ndrange=(n, n))
        KernelAbstractions.synchronize(Backend)
    end
    bench
end

plot_matmul(matmul_sizes, matmul_results)



## 2.4. Performance and memory levels

`MatrixMultiplication_kernel!` spent almost all of its time reading `DA` and `DB` from global memory. Speeding that kernel up means moving values into a closer memory before the threads reuse them. The picture shows which threads that closer memory belongs to.

![A grid of thread blocks, with one block opened to show the threads inside it](img/threadsandblocks.webp)

#### 2.4.1. Shared memory

The green box is the grid of a launch: the blocks of a kernel such as `MatrixMultiplication_kernel!`. The yellow zoom opens one of those blocks, Block (1, 1), down to its threads. The picture numbers blocks and threads from 0. `@index` in KernelAbstractions numbers them from 1.

Each squiggle is one thread, the same kind of thread that computed one element in `vadd_kernel!` and one output entry in `MatrixMultiplication_kernel!`. The threads drawn inside a single yellow block are the ones scheduled together on one streaming multiprocessor.

**Shared memory** is a small region on that SM, and it belongs to the block. Every thread inside Block (1, 1) can read and write it. Block (0, 0) has a separate region, and its threads cannot see this one. KernelAbstractions calls this local memory and allocates it with `@localmem`. A thread that stores a value there has to be finished before another thread in the block reads it, which is what `@synchronize` waits for.

`vadd_kernel!` never needed that region: each thread touched only its own `a[i]` and `b[i]`. `MatrixMultiplication_kernel!` does. One element of `DA` is reused down a column of the 32×32 tile, and one element of `DB` across a row. Loaded once into the block's shared memory, those values serve the whole tile. In the kernel we timed, each product fetched them again from global memory.

#### 2.4.2. The levels, from the thread outward

- **Registers** belong to one thread, one squiggle in the yellow block. `sum`, `row`, and `col` in `MatrixMultiplication_kernel!` live here. This is the fastest memory, and each thread has very little of it.
- **Shared memory** belongs to one block, the yellow box. It sits on the SM next to those threads. It is larger than one thread's registers and much faster than device DRAM, and it is the scratch space a tiled matrix multiply uses.
- **Global memory** belongs to the whole green grid. `DA`, `DB`, and `DC` live here, in device DRAM. Every block of the launch can read and write it, and a round trip to it is the slowest of the three.

The gap between `MatrixMultiplication_kernel!` and `DA * DB` is mostly traffic to that last level. A faster kernel loads a tile of `DA` and `DB` into the shared memory of each block, waits until every thread in the block has finished that load, and then accumulates many products from registers and shared memory. The global read happens once per tile instead of once per product.



## 2.5. Shared memory for matrix multiplication

`MatrixMultiplication_kernel!` already gives one thread one element of `DC`. That thread walks the whole inner dimension and reads `DA` and `DB` from global memory on every step. We keep the same output element. We change the walk, so each value of `DA` and `DB` is loaded once for the block and then reused.

Build it in this order:

- *Give the block a 16×16 tile of `DC`.*  
  A block of 32×32 threads is 1024 threads, the largest workgroup we used for `vadd_kernel!`. Some devices allow fewer than that, so this block is 16×16: 256 threads. Our matrices are 1024 on a side, and 1024 divides by 16, so every block is full. The block owns a 16×16 patch of `DC`.

- *Name this thread's element from the block.*  
  `@index(Group)` is which patch this block owns. `@index(Local)` is this thread's place inside the 16×16 patch. The element is

  `I = (gi - 1) * 16 + i`

  `J = (gj - 1) * 16 + j`

  That is the same pair `@index(Global)` produced in `MatrixMultiplication_kernel!`.

- *Cut the inner loop into tiles of 16.*  
  The inner dimension is 1024. Instead of one loop of 1024 global reads, take it 16 columns at a time. Each step needs a 16×16 piece of `DA` (the rows this block owns, and the current 16 columns) and a 16×16 piece of `DB` (those same columns, and the columns of `DC` this block owns). The block has exactly 256 threads, so each thread can fetch one element of each piece.

- *Put those pieces in the block's shared memory.*  
  Each block has the shared-memory region from the picture. Allocate two arrays there, one for the piece of `DA` and one for the piece of `DB`. Each array is 16×16, one element for every thread in the block.

- *Load, then wait.*  
  Each thread writes its element of `DA` into the first tile and its element of `DB` into the second. A thread whose index falls outside the matrix writes a zero, so a short tile still multiplies. Then `@synchronize`. Until every thread in the yellow block has finished those stores, another thread's shared-memory write is not visible yet.

- *Multiply from the tile, and keep a private sum.*  
  Now the thread loops `k` from 1 to 16 and multiplies `tile1[i, k] * tile2[k, j]`. Those reads hit shared memory, not `DA` and `DB`. The sum of that tile has to still be there after the next wait, and a plain local variable does not survive `@synchronize`. Give the thread one private slot, `outval[1]`, and add the tile's product into it.

- *Wait, then take the next tile.*  
  A second `@synchronize` runs before the next step of the inner dimension. It keeps any thread from overwriting a shared tile while another thread is still multiplying from it. Then the block loads the next 16 columns into the same two arrays, and the thread adds another product into `outval[1]`.

- *Write the element.*  
  When every tile has been added, the thread stores `outval[1]` into `DC[I, J]`, if `I` and `J` lie inside the matrix.

The kernel below follows that list in order.



In [ ]:
# Global element from the block and local index (same idea as @index(Global)).
# input1 is DA and input2 is DB when we launch on those matrices.
@kernel function coalesced_matmul_kernel!(output, @Const(input1), @Const(input2))
    gi, gj = @index(Group, NTuple)
    i, j = @index(Local, NTuple)

    # 16, once we compile with a (16, 16) workgroup. @uniform keeps it visible across the waits.
    TILE_DIM = @uniform @groupsize()[1]

    # Two shared tiles for this block, one element per thread.
    tile1 = @localmem eltype(output) (TILE_DIM, TILE_DIM)
    tile2 = @localmem eltype(output) (TILE_DIM, TILE_DIM)

    # Private sum per thread, so it survives @synchronize.
    outval = @private eltype(output) 1
    @inbounds outval[1] = zero(eltype(output))

    @uniform N = size(output, 1)
    @uniform R = size(input1, 2)
    @uniform M = size(output, 2)
    # How many tiles of length TILE_DIM cover the inner dimension.
    @uniform NUM_TILES = div(R + TILE_DIM - 1, TILE_DIM)

    for t in 0:(NUM_TILES - 1)
        I = (gi - 1) * TILE_DIM + i
        J = (gj - 1) * TILE_DIM + j

        # Load one element of DA and one of DB. Past the edge, store zero.
        if I <= N && t * TILE_DIM + j <= R
            @inbounds tile1[i, j] = input1[I, t * TILE_DIM + j]
        else
            @inbounds tile1[i, j] = zero(eltype(output))
        end
        if t * TILE_DIM + i <= R && J <= M
            @inbounds tile2[i, j] = input2[t * TILE_DIM + i, J]
        else
            @inbounds tile2[i, j] = zero(eltype(output))
        end

        # The rest of the block must finish the stores before anyone multiplies.
        @synchronize

        # This tile's product, from shared memory, added into the private sum.
        out = zero(eltype(output))
        for k in 1:TILE_DIM
            @inbounds out += tile1[i, k] * tile2[k, j]
        end
        @inbounds outval[1] += out

        # Finish reading before the next tile reuses tile1 and tile2.
        @synchronize
    end

    # The indices again, then the one write to DC.
    I = (gi - 1) * TILE_DIM + i
    J = (gj - 1) * TILE_DIM + j
    if I <= N && J <= M
        @inbounds output[I, J] = outval[1]
    end
end



*Launch, check, and plot.* Compile it for a 16×16 workgroup and launch it on the same `DA`, `DB`, and `DC`. The `ndrange` is rounded up to a multiple of 16 so the last block still runs all 256 threads. Those extra threads fill the shared tile with zeros, and the bounds check keeps them from writing past the end of `DC`. On these 1024×1024 matrices the rounded range is the matrix itself.

Then compare `DC` with `DA * DB`.



In [ ]:
TILE_DIM = 16
tiled! = coalesced_matmul_kernel!(Backend, (TILE_DIM, TILE_DIM))
ndrange = (
    cld(size(DC, 1), TILE_DIM) * TILE_DIM,
    cld(size(DC, 2), TILE_DIM) * TILE_DIM,
)
tiled!(DC, DA, DB, ndrange=ndrange)
KernelAbstractions.synchronize(Backend)

isapprox(DC, DA * DB)



Time the library product and the tiled kernel the same way as the pair above. The kernel benchmark includes `KernelAbstractions.synchronize`, so it waits until the device finishes. The naive `MatrixMultiplication_kernel!` timing is the benchmark just before the memory section.



In [ ]:
@benchmark DA * DB



In [ ]:
@benchmark begin
    tiled!(DC, DA, DB, ndrange=ndrange)
    KernelAbstractions.synchronize(Backend)
end



Read the three timings together. `MatrixMultiplication_kernel!`, timed just before the memory section, reads `DA` and `DB` from global memory on every product. `coalesced_matmul_kernel!` reads each element once per tile into the block's shared memory, and the thread keeps the sum in `outval`. `DA * DB` is still ahead of both when the backend library uses a tuned kernel. The difference between the two kernels we wrote is the shared-memory tile.

The same tiling is the KernelAbstractions example [`performant_matmul.jl`](https://github.com/JuliaGPU/KernelAbstractions.jl/blob/main/examples/performant_matmul.jl).



One more series, `coalesced_matmul_kernel!`, with the 16×16 blocks from the steps above. Every size in `matmul_sizes` is a multiple of 16, so each block is full. `plot_matmul` adds the line to the same figure.



In [ ]:
benchmark_matmul!(matmul_results, "GPU coalesced_matmul_kernel!", matmul_sizes) do n
    A = rand(n, n)
    B = rand(n, n)
    DA = copyto!(allocate(Backend, Float64, n, n), A)
    DB = copyto!(allocate(Backend, Float64, n, n), B)
    DC = KernelAbstractions.zeros(Backend, Float64, n, n)
    k! = coalesced_matmul_kernel!(Backend, (16, 16))
    function bench()
        k!(DC, DA, DB, ndrange=(n, n))
        KernelAbstractions.synchronize(Backend)
    end
    bench
end

plot_matmul(matmul_sizes, matmul_results)



# 3. Tutorial completed

Congratulations — you finished the matrix-multiplication notebook.

Along the way you:

- reused the same portable `Backend` setup as the GPU introduction
- implemented matrix multiplication on the CPU and as a naive GPU kernel
- timed CPU and GPU versions across matrix sizes and plotted the results
- learned the memory hierarchy (registers, shared memory, global memory) and rebuilt the multiply so each block tiles `DA` and `DB` through shared memory

## Optional next steps

Nothing to turn in — if you want to keep playing after the session, here are some ideas:

1. **Bank conflicts and padding.** Shared memory is split into banks. When many threads in a warp hit the same bank, those reads serialize. Look up GPU shared-memory bank conflicts, then try padding the tile arrays (for example `(TILE_DIM + 1, TILE_DIM)`) and see whether your device gets faster.
2. **Tile and workgroup size.** Try `TILE_DIM` of 8, 16, and 32. Watch occupancy, shared-memory use, and the size plot. Is bigger always better?
3. **Registers.** Registers are the fastest memory in the hierarchy from section 2.4. Keep more of the inner product in registers (for example a small register tile of several `C` values per thread, or holding reused `A`/`B` values in locals across the `k` loop) so you touch shared and global memory less often. Watch that using too many registers can lower occupancy.
4. **More work per thread.** Have each thread own several output elements, which pairs naturally with register tiling above, before writing `DC`.
5. **Vendor libraries and docs.** Compare your best kernel to `DA * DB` (cuBLAS / rocBLAS / Metal). Skim a GPU performance guide for your backend and notice which of its advice your kernel still ignores.

If you try something, reuse `benchmark_matmul!` and `plot_matmul` to see the effect. The goal is curiosity, not a finished report.

